In [1]:
import os
import sys

import re
import pandas as pd
import networkx as nx
import tqdm
import random
import pickle
import json
import subprocess
import numpy as np
import httpx
import requests
import time

from dotenv import load_dotenv
from tqdm import tqdm

# Determine the project root directory for relative imports
try:
    # This will work in scripts where __file__ is defined
    current_dir = os.path.dirname(os.path.abspath(__file__))
    # Assuming "src" is parallel to the script folder
    project_root = os.path.abspath(os.path.join(current_dir, ".."))
except NameError:
    # In notebooks __file__ is not defined: assume we're in notebooks/riziv_dataset/
    project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Local application imports

#from main.ollama_utils import get_ollama_embedding


In [2]:
load_dotenv(os.path.join(project_root, ".env"))
response_gpu_temp = httpx.get(f"{os.getenv('NGROK_URL')}/health/gpu")

response_gpu_temp

<Response [200 OK]>

In [3]:
# Define the path to the BSARD dataset files
BSARD_data_path = os.path.join(project_root, "data", "BSARD_dataset")

In [4]:
with open(os.path.join(BSARD_data_path,"intermediate", "hybrid_graph_full_B.pkl"), 'rb') as f:
    G = pickle.load(f)

In [5]:
NGROK_URL = os.getenv("NGROK_URL")
API_TOKEN = os.getenv("API_KEY")

def get_ollama_embedding(input_text):

    response = requests.post(
        f"{NGROK_URL}/embed",
        headers={
            "Authorization": f"Bearer {API_TOKEN}",
                },
        json={
            "input": input_text,
            }
        )
    if len(response.json()["embeddings"]) == 1:

        return response.json()["embeddings"][0]
    
    else:

        return response.json()["embeddings"]


In [6]:
#len(get_ollama_embedding(["1.1.1.1"]))

In [8]:
G.nodes['1.1.1']["text"]

"Le présent Code règle une matière visée à l'article 39 de la Constitution."

In [9]:
# Configuración
BATCH_SIZE = 500

# 1) Extraer nodos de tipo 'Article'
article_nodes = [
    n for n, d in G.nodes(data=True)
    if d.get("node_type") == "Article"
]

# 2) Preparar diccionario para embeddings
embeddings_dict = {node: None for node in article_nodes}

# 3) Generar embeddings en lotes
pbar = tqdm(total=len(article_nodes), desc="Generating embeddings")
for i in range(0, len(article_nodes), BATCH_SIZE):
    batch_nodes = article_nodes[i : i + BATCH_SIZE]
    batch_texts = [G.nodes[node]["text"] for node in batch_nodes]

    # Llamada a la API de embeddings
    batch_embeddings = get_ollama_embedding(batch_texts)

    # Asignar embeddings al diccionario
    for node, emb in zip(batch_nodes, batch_embeddings):
        embeddings_dict[node] = emb

    # Actualizar barra de progreso
    pbar.update(len(batch_nodes))

    # Guardar progreso intermedio
    file_path = os.path.join(BSARD_data_path, "intermediate", "article_node_embeddings.pkl")
    with open(file_path, 'wb') as f:
        pickle.dump(embeddings_dict, f)

    # Pausa breve para no saturar la API
    time.sleep(0.5)

pbar.close()

# 4) Guardar resultados finales
final_path = os.path.join(BSARD_data_path, "intermediate", "article_node_embeddings_final_B.pkl")
with open(final_path, 'wb') as f:
    pickle.dump(embeddings_dict, f)

#print("Embeddings generados y guardados en:", final_path)

Generating embeddings:   0%|          | 0/22621 [00:00<?, ?it/s]

Generating embeddings: 100%|██████████| 22621/22621 [08:38<00:00, 43.63it/s]


In [10]:
nx.set_node_attributes(G, embeddings_dict, "embedding")

In [11]:
BATCH_SIZE = 500

# 1) Extraer nodos de tipo 'KeyTerm'
keyterm_nodes = [
    n for n, d in G.nodes(data=True)
    if d.get("node_type") == "KeyTerm"
]

# 2) Preparar diccionario para embeddings
embeddings_dict = {node: None for node in keyterm_nodes}

# 3) Generar embeddings en lotes
pbar = tqdm(total=len(keyterm_nodes), desc="Embedding KeyTerm")
for i in range(0, len(keyterm_nodes), BATCH_SIZE):
    batch_nodes = keyterm_nodes[i : i + BATCH_SIZE]
    # Usamos el ID del nodo (convertido a str) como texto de entrada
    batch_texts = [str(node) for node in batch_nodes]

    # Llamada a la API de embeddings
    batch_embeddings = get_ollama_embedding(batch_texts)

    # Asignar embeddings al diccionario
    for node, emb in zip(batch_nodes, batch_embeddings):
        embeddings_dict[node] = emb

    # Actualizar barra de progreso
    pbar.update(len(batch_nodes))

    # Guardar progreso intermedio
    checkpoint = os.path.join(BSARD_data_path, "keyterm_node_embeddings_B.pkl")
    with open(checkpoint, "wb") as f:
        pickle.dump(embeddings_dict, f)

    time.sleep(0.5)  # pausa breve para no saturar la API

pbar.close()

# 4) Guardar resultados finales
final_fp = os.path.join(BSARD_data_path, "keyterm_node_embeddings_final_B.pkl")
with open(final_fp, "wb") as f:
    pickle.dump(embeddings_dict, f)

#print("Embeddings de KeyTerm generados y guardados en:", final_fp)

# 5) (Opcional) Asignar el atributo 'embedding' en el grafo
nx.set_node_attributes(G, embeddings_dict, "embedding")

Embedding KeyTerm: 100%|██████████| 1283/1283 [00:20<00:00, 63.68it/s]


In [12]:
# 1) Pre-count the "Act" nodes so tqdm knows the total
act_nodes = [
    (n, d) for n, d in G.nodes(data=True) if d.get("node_type") == "Act"
]

# 2) Progress bar: one tick per generated embedding
for node, data in tqdm(act_nodes,
                       desc="Generating embeddings",
                       total=len(act_nodes)):      # optional: tqdm can infer it
    data["embedding"] = get_ollama_embedding(data["act_title"])
    time.sleep(0.5)

Generating embeddings: 100%|██████████| 35/35 [00:25<00:00,  1.39it/s]


In [13]:
# 1) Embeddings for Title: average of its Articles
title_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "Title"]
for title in tqdm(title_nodes, desc="Aggregating Title embeddings"):
    # extract only Articles with embedding
    child_articles = [
        nbr for nbr in G.successors(title)
        if G.nodes[nbr].get("node_type") == "Article"
           and "embedding" in G.nodes[nbr]
    ]
    if not child_articles:
        continue

    embs = np.stack([G.nodes[art]["embedding"] for art in child_articles])
    G.nodes[title]["embedding"] = embs.mean(axis=0).tolist()


# 2) Embeddings for Book: average of its Articles or Titles
book_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "Book"]
for book in tqdm(book_nodes, desc="Aggregating Book embeddings"):
    # children can be Articles or Titles
    child_embs = []
    for nbr in G.successors(book):
        nt = G.nodes[nbr].get("node_type")
        if ("embedding" in G.nodes[nbr]) and nt in ("Article", "Title"):
            child_embs.append(G.nodes[nbr]["embedding"])

    if not child_embs:
        continue

    embs = np.stack(child_embs)
    G.nodes[book]["embedding"] = embs.mean(axis=0).tolist()


# 3) Embeddings for Act: mix of its own embedding and the average of its neighbors
act_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "Act"]
for act in tqdm(act_nodes, desc="Aggregating Act embeddings"):
    # neighbors with embedding (could be Book or Title depending on your graph)
    nbr_embs = [
        G.nodes[nbr]["embedding"]
        for nbr in G.successors(act)
        if "embedding" in G.nodes[nbr]
    ]
    if not nbr_embs or "embedding" not in G.nodes[act]:
        continue

    nbr_mean = np.stack(nbr_embs).mean(axis=0)
    own_emb  = np.array(G.nodes[act]["embedding"])
    # here we choose weight 0.5/0.5; adjust if you want a different balance
    mixed = (own_emb + nbr_mean) / 2
    G.nodes[act]["embedding"] = mixed.tolist()

# 4) Embeddings for Central node: average of all its neighbors
meta_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "Central Node"]

for meta in tqdm(meta_nodes, desc="Aggregating Meta embeddings"):
    # incoming and outgoing neighbors, without duplicates
    neighs = set(G.successors(meta)).union(G.predecessors(meta))
    neigh_embs = [G.nodes[v]["embedding"] for v in neighs if "embedding" in G.nodes[v]]

    if not neigh_embs:
        continue  # no embeddings available

    embs = np.stack(neigh_embs)
    G.nodes[meta]["embedding"] = embs.mean(axis=0).tolist()

Aggregating Title embeddings: 0it [00:00, ?it/s]
Aggregating Act embeddings: 100%|██████████| 35/35 [00:00<00:00, 3050.53it/s]
Aggregating Meta embeddings: 0it [00:00, ?it/s]


In [14]:
# Ensure all nodes have an attribute "embedding"
assert all("embedding" in data for _, data in G.nodes(data=True)), "Some nodes lack an embedding!"


In [15]:
with open(os.path.join(BSARD_data_path, 'hybrid_graph_full_wsem_B.pkl'), 'wb') as f:
    pickle.dump(G, f)